# A delayed coker

`difflow_refinery.coking` models a delayed coker. Its yields come from the Gary-Handwerk correlations in Conradson carbon (CCR), and its product gravities follow from the correlations' volume yields. The unit balances sulfur, nitrogen and metals, and it pools several residue feeds before the yield model. This notebook:

1. runs the unit on a blend of vacuum residue and FCC slurry;
2. differentiates the yields with respect to feed CCR and the operating levers;
3. builds a planning block;
4. compares the AD sensitivities with the published delta-base vectors of the Du et al. (2025) Case 2 cokers, when the benchmark data are available.

Constants other than the yield correlation are **illustrative**; see the docs.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
from difflow_refinery.coking import PRODUCTS, CokerFeed, DelayedCoker, coker_block

jax.config.update("jax_enable_x64", True)

vr = CokerFeed.from_properties(8.0, sg=1.03, sulfur_wt=3.5, ccr_wt=22.0,
                               nitrogen_wppm=4000.0, nickel_vanadium_wppm=250.0)
slurry = CokerFeed.from_properties(2.0, sg=1.08, sulfur_wt=1.2, ccr_wt=6.0)
unit = DelayedCoker()
out = unit.solve([vr, slurry])
for k in PRODUCTS:
    print(f"{k:8s} {100 * float(out['yield.' + k]):6.2f} wt%")
print("naphtha SG", round(float(out["naphtha.SG"]), 4), " coke S wt%", round(float(out["coke.S_wt"]), 2),
      " coke Ni+V wppm", round(float(out["coke.NiV_wppm"]), 0))
print("balances", {k: float(v) for k, v in out.items() if k.startswith("balance.")})

dry_gas    6.20 wt%
lpg        3.34 wt%
h2s        0.97 wt%
nh3        0.00 wt%
naphtha   17.74 wt%
diesel    20.84 wt%
gas_oil   20.84 wt%
coke      30.08 wt%
naphtha SG 0.7587  coke S wt% 3.03  coke Ni+V wppm 665.0
balances {'balance.mass': 0.0, 'balance.S': 0.0, 'balance.N': 0.0, 'balance.NiV': 0.0}


## Sensitivities

The yields are differentiable with respect to the feed and the operating levers. Each column of the Jacobian below is a delta vector of the kind a planner uses.

In [2]:
def y(x):   # (CCR wt%, drum pressure barg, recycle ratio) -> yields
    feed = CokerFeed.from_properties(1.0, sg=1.03, sulfur_wt=3.5, ccr_wt=x[0])
    return unit.yields(feed, drum_P=x[1], recycle=x[2])

J = jax.jacfwd(y)(jnp.array([22.0, 1.03, 0.05]))
print(f"{'product':8s} {'d/dCCR':>9s} {'d/dP':>9s} {'d/dR':>9s}")
for k, row in zip(PRODUCTS, np.asarray(J)):
    print(f"{k:8s} " + " ".join(f"{v:9.5f}" for v in row))

block = coker_block(unit, CokerFeed.blend([vr, slurry]), ["feed.ccr", "drum_P", "recycle"])
print(block.y_names)

product     d/dCCR      d/dP      d/dR
dry_gas    0.00094   0.00214   0.02852
lpg        0.00050   0.00115   0.01536
h2s        0.00000   0.00000   0.00000
nh3        0.00000   0.00000   0.00000
naphtha    0.00343   0.00000   0.00000
diesel    -0.01043  -0.00869  -0.12754
gas_oil   -0.01043  -0.00869  -0.12754
coke       0.01600   0.01408   0.21120
['yield.dry_gas', 'yield.lpg', 'yield.h2s', 'yield.naphtha', 'yield.diesel', 'yield.gas_oil', 'yield.coke', 'naphtha.SG', 'gas_oil.SG', 'coke.S_wt']


## Against the Du et al. (2025) Case 2 cokers

Case 2 of the benchmark has two delta-base delayed cokers. Below are our yields and AD sensitivities at their base feeds, next to the published ones. Coke and its CCR slope agree, and so does H2S and its sulfur slope. Naphtha and gas move with CCR in opposite directions. The correlation has no SG term, so all its SG sensitivities are zero. The benchmark data are downloaded, not shipped; without them this cell only says so.

In [3]:
from difflow_refinery.benchmarks import du2025
from difflow_refinery.benchmarks.du2025.calibration import unit_base

case2 = du2025.find_case(2)
if case2 is None:
    print("Case 2 data not available (du2025.fetch(cases=(2,)) downloads them)")
else:
    data = du2025.load_case(case2)
    b = unit_base(data, "Upd0")
    outlets = {"coke": "s272", "naphtha": "s276", "h2s": "s274"}
    x0 = jnp.array([b.props["Q0"], b.props["Q1"], b.props["Q22"]])     # SG, S, CCR
    f = lambda x: unit.yields(CokerFeed.from_properties(1.0, sg=x[0], sulfur_wt=x[1], ccr_wt=x[2]))
    ours, Jc = np.asarray(f(x0)), np.asarray(jax.jacfwd(f)(x0))
    for k, s in outlets.items():
        i, j = PRODUCTS.index(k), b.outlets.index(s)
        pub = [b.delta[q][j] for q in ("Q0", "Q1", "Q22")]
        print(f"{k:8s} yield {ours[i]:.4f} (published {b.yields[j]:.4f});  "
              f"d/d(SG, S, CCR) ours {np.round(Jc[i], 4)}  published {np.round(pub, 4)}")

coke     yield 0.3120 (published 0.3100);  d/d(SG, S, CCR) ours [0.    0.    0.016]  published [0.018  0.     0.0156]
naphtha  yield 0.1798 (published 0.1720);  d/d(SG, S, CCR) ours [0.     0.     0.0034]  published [-0.018  -0.0003 -0.0073]
h2s      yield 0.0096 (published 0.0020);  d/d(SG, S, CCR) ours [0.     0.0032 0.    ]  published [ 0.003   0.0035 -0.0002]
